In [ ]:
import zipfile
import os


zip_file_path = '/content/archive.zip'

extract_dir = '/content/glass_dataset'

os.makedirs(extract_dir, exist_ok=True)


with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Files extracted to: {extract_dir}")

print("Extracted files:")
for root, dirs, files in os.walk(extract_dir):
    for file in files:
        print(os.path.join(root, file))


Files extracted to: /content/glass_dataset
Extracted files:
/content/glass_dataset/glass.csv


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np


data_path = os.path.join(extract_dir, 'glass.csv')
df = pd.read_csv(data_path)

print("Dataset Head:")
display(df.head())
print("\nDataset Info:")
df.info()

X = df.drop('Type', axis=1).values
y = (df['Type'] == 1).astype(int).values


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Class distribution in y_train (0s/1s): {np.bincount(y_train)}")

Dataset Head:


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214 entries, 0 to 213
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RI      214 non-null    float64
 1   Na      214 non-null    float64
 2   Mg      214 non-null    float64
 3   Al      214 non-null    float64
 4   Si      214 non-null    float64
 5   K       214 non-null    float64
 6   Ca      214 non-null    float64
 7   Ba      214 non-null    float64
 8   Fe      214 non-null    float64
 9   Type    214 non-null    int64  
dtypes: float64(9), int64(1)
memory usage: 16.8 KB

Shape of X_train: (171, 9)
Shape of y_train: (171,)
Class distribution in y_train (0s/1s): [115  56]


In [ ]:
class LogisticRegressionPerceptron:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None

    # Sigmoid activation function
    def _sigmoid(self, z):

        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    # Initialize weights and bias
    def _initialize_parameters(self, n_features):
        self.weights = np.zeros(n_features)
        self.bias = 0

    # Forward pass: calculate predictions
    def _predict_proba(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted = self._sigmoid(linear_model)
        return y_predicted

    # Binary Cross-Entropy Loss
    def _compute_loss(self, y_true, y_pred):
        epsilon = 1e-10 # Small value to prevent log(0)
        loss = -np.mean(y_true * np.log(y_pred + epsilon) + (1 - y_true) * np.log(1 - y_pred + epsilon))
        return loss

    # Gradient computation and parameter update
    def _update_parameters(self, X, y_true, y_pred):
        m = X.shape[0] # Number of samples

        # Gradients
        dw = (1 / m) * np.dot(X.T, (y_pred - y_true))
        db = (1 / m) * np.sum(y_pred - y_true)

        # Update parameters
        self.weights -= self.learning_rate * dw
        self.bias -= self.learning_rate * db

    # Training function
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self._initialize_parameters(n_features)
        self.losses = []

        for _ in range(self.n_iterations):

            y_predicted = self._predict_proba(X)

            loss = self._compute_loss(y, y_predicted)
            self.losses.append(loss)


            self._update_parameters(X, y, y_predicted)

        print(f"Training complete. Final loss: {self.losses[-1]:.4f}")

    def predict(self, X):
        y_predicted_proba = self._predict_proba(X)
        y_predicted_cls = (y_predicted_proba >= 0.5).astype(int)
        return y_predicted_cls


model = LogisticRegressionPerceptron(learning_rate=0.1, n_iterations=10000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)

acc = accuracy(y_test, y_pred)
print(f"\nModel Accuracy on Test Set: {acc:.4f}")

Training complete. Final loss: 0.4340

Model Accuracy on Test Set: 0.7442
